- takes into account: tld distribution, length, ...

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..")))

from data.utils.general import get_sld
pd.set_option("display.float_format", "{:.2f}".format)

In [ ]:
long_dataset = pd.read_csv("dataset_long.csv")
short_dataset = pd.read_csv("collected_urls_low_length.csv")

long_dataset = long_dataset.drop_duplicates(subset=["url"])
short_dataset = short_dataset.drop_duplicates(subset=["url"])

In [ ]:
display(long_dataset.info())
display(short_dataset.info())

In [ ]:
long_dataset["sld"] = long_dataset["url"].apply(get_sld)
short_dataset["sld"] = short_dataset["url"].apply(get_sld)

long_dataset["url_length"] = long_dataset["url"].str.len()
short_dataset["url_length"] = short_dataset["url"].str.len()

In [ ]:
# Limit to at most 50 URLs per SLD (randomly if there are more)
max_sld_count = 150
seed = 42  # set for reproducibility

print(f"Short dataset size before filtering: {len(short_dataset)}")

# Shuffle once, then take the first 50 of each SLD group
short_dataset_filtered = (
    short_dataset.sample(frac=1, random_state=seed)  # global shuffle for randomness
    .groupby("sld", group_keys=False)
    .head(max_sld_count)  # keep at most 50 per SLD
)

print(f"Short dataset size after filtering: {len(short_dataset_filtered)} / {len(short_dataset)}")
print(f"Number of unique SLDs after filtering: {short_dataset_filtered['sld'].nunique()} / {short_dataset['sld'].nunique()}")


# long_dataset_filtered = long_dataset.sample(frac=1, random_state=seed).groupby("sld", group_keys=False).head(max_sld_count)

# print(f"Long dataset size after filtering: {len(long_dataset_filtered)} / {len(long_dataset)}")
# print(f"Number of unique SLDs after filtering: {long_dataset_filtered['sld'].nunique()} / {long_dataset['sld'].nunique()}")

In [ ]:
length_table_stats = pd.DataFrame()
# mean, ... columns, long short as rows
length_table_stats = pd.DataFrame({
    'long_dataset': long_dataset["url_length"],
    'short_dataset': short_dataset["url_length"]
})

print(length_table_stats.describe())



In [ ]:
combined = pd.concat([long_dataset, short_dataset_filtered], ignore_index=True)

In [ ]:
import json

def load_tld_targets(path: str, min_pct: float = 1.0):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    total_counts = data["total"]
    total_sum = sum(total_counts.values())
    if total_sum == 0:
        return {'total_count': 0, 'tlds': {'__other__': {'count': 0, 'pct': 0.0}}, 'order': ['__other__']}

    keep = {}
    other_count = 0
    for tld, cnt in total_counts.items():
        if tld == "__blank__":
            other_count += cnt
            continue
        pct = cnt / total_sum * 100.0
        if pct >= min_pct:
            keep[tld] = {'count': cnt, 'pct': pct}
        else:
            other_count += cnt

    other_pct = other_count / total_sum * 100.0
    keep["__other__"] = {'count': other_count, 'pct': other_pct}

    order = sorted([k for k in keep.keys() if k != "__other__"],
                   key=lambda k: keep[k]['count'], reverse=True) + ["__other__"]

    return {'total_count': total_sum, 'tlds': keep, 'order': order}


targets = load_tld_targets("tld_counts_by_dataset.json", min_pct=0.5)
print(targets)

In [ ]:
import numpy as np
import pandas as pd
import tldextract

def ensure_tld_column(df: pd.DataFrame, url_col: str = "url", tld_col: str = "_tld") -> pd.DataFrame:
    """Adds/overwrites df[tld_col] with parsed TLDs; blanks → '__blank__'."""
    if tld_col in df.columns:
        return df
    ext = df[url_col].astype("string").map(lambda x: tldextract.extract(x if isinstance(x, str) else "").suffix)
    out = df.copy()
    out[tld_col] = ext.mask(ext.eq(""), "__blank__")
    return out

def _hamilton_round(target_fracs: np.ndarray, total: int) -> np.ndarray:
    """Largest-remainder rounding to integer counts that sum to total."""
    raw = target_fracs * total
    base = np.floor(raw).astype(int)
    rem = total - base.sum()
    if rem > 0:
        bump_idx = np.argsort(raw - base)[::-1][:rem]
        base[bump_idx] += 1
    return base

def sample_to_match_tld_distribution(
    combined_df: pd.DataFrame,
    targets: dict,                 # output of load_tld_targets(...)
    url_col: str = "url",
    tld_col: str = "_tld",
    size: int | None = None,
    seed: int | None = 42,
) -> pd.DataFrame:
    """
    Build a sample from combined_df whose TLD mix follows 'targets' (counts/percentages),
    with '__other__' drawn from all rows whose TLD not in the specific target keys.
    """
    rng = np.random.default_rng(seed)
    df = ensure_tld_column(combined_df, url_col=url_col, tld_col=tld_col)

    order = targets['order']                   # specific TLDs + '__other__' at the end
    tld_info = targets['tlds']
    specific_tlds = [t for t in order if t != "__other__"]
    all_target_tlds = set(specific_tlds)

    # Desired sample size
    if size is None:
        size = len(df)

    # Desired counts by target percentages
    target_pcts = np.array([tld_info[t]['pct'] for t in specific_tlds] + [tld_info['__other__']['pct']], dtype=float)
    target_fracs = target_pcts / target_pcts.sum()
    desired_counts = _hamilton_round(target_fracs, size)
    desired_spec = desired_counts[:-1]   # specific TLDs
    desired_other = int(desired_counts[-1])

    # Build index pools per TLD (vectorized boolean indexing and np.where)
    tld_values = df[tld_col].to_numpy()
    indices = np.arange(len(df))

    pools = {}
    for t in specific_tlds:
        mask = (tld_values == t)
        pools[t] = indices[mask]

    # __other__ pool: everything NOT in specific_tlds
    if specific_tlds:
        spec_mask = np.isin(tld_values, specific_tlds)
        other_pool = indices[~spec_mask]
    else:
        other_pool = indices  # if no specific TLDs

    # Satisfy specific TLD buckets first; track selected ids
    selected = []

    # Capacity-aware allocation with simple one-pass cap and deficit reallocation
    # 1) initial take (cap by availability)
    avail_spec = np.array([pools[t].size for t in specific_tlds], dtype=int)
    take_spec = np.minimum(desired_spec, avail_spec)

    # 2) reallocate deficits to other specific TLDs with spare capacity, proportional to target
    deficit = int(desired_spec.sum() - take_spec.sum())
    if deficit > 0:
        capacity = avail_spec - take_spec
        while deficit > 0 and capacity.sum() > 0:
            weights = (target_fracs[:-1] * (capacity > 0))
            s = weights.sum()
            if s == 0:
                # uniform fallback
                weights = (capacity > 0).astype(float)
                s = weights.sum()
            share = weights / s * deficit
            incr = np.floor(share).astype(int)
            rem = deficit - incr.sum()
            if rem > 0:
                bump_idx = np.argsort(share - incr)[::-1][:rem]
                incr[bump_idx] += 1
            incr = np.minimum(incr, capacity)
            take_spec += incr
            capacity -= incr
            deficit = int(desired_spec.sum() - take_spec.sum())

    # 3) sample per specific TLD
    for t, k in zip(specific_tlds, take_spec):
        if k <= 0:
            continue
        pool = pools[t]
        if k >= pool.size:
            chosen = pool
        else:
            chosen = rng.choice(pool, size=k, replace=False)
        selected.append(chosen)

    # 4) __other__: desired count = desired_other + any leftover deficit from specifics
    # Remaining capacity in other_pool:
    already = np.zeros(len(df), dtype=bool)
    if selected:
        already[np.concatenate(selected)] = True
    other_candidates = other_pool[~already[other_pool]]

    # Any unresolved deficit from specifics (if not fully reallocated) gets added to other
    unresolved_deficit = int(desired_spec.sum() - take_spec.sum())
    desired_other_final = max(0, desired_other + unresolved_deficit)

    if desired_other_final > 0:
        k = min(desired_other_final, other_candidates.size)
        if k > 0:
            other_chosen = rng.choice(other_candidates, size=k, replace=False)
            selected.append(other_chosen)

    if not selected:
        # nothing could be sampled; return empty
        return df.iloc[[]].copy()

    final_idx = np.concatenate(selected)
    # If we by chance exceeded 'size' (shouldn't happen), trim
    if final_idx.size > size:
        final_idx = rng.choice(final_idx, size=size, replace=False)

    out = df.loc[final_idx].sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out


out = sample_to_match_tld_distribution(combined, targets, size=2000000)

In [ ]:

import tldextract
def get_tld(url: str) -> str:
    suf = tldextract.extract(url if isinstance(url, str) else "").suffix
    return suf if suf else "__blank__"   # keep blanks explicit
out["tld"] = out["url"].map(get_tld)

In [ ]:

import pandas as pd


def print_tld_value_counts(
    df: pd.DataFrame,
    url_col: str = "url",
    tld_col: str = "tld",
    min_pct: float = 0.5,
    include_blank_in_others: bool = True,
    pct_decimals: int = 2,
) -> pd.DataFrame:
    """
    Adds/overwrites df[tld_col], computes counts & percentages for TLDs,
    folds anything < min_pct into 'Others' (and optionally '__blank__'),
    sorts by count desc with Others at the end, prints nicely, and
    returns the final table as a DataFrame.
    """
    # Ensure TLD column
    df = df.copy()
    df[tld_col] = df[url_col].map(get_tld)

    # Counts / %
    counts = df[tld_col].value_counts(dropna=False)
    total = counts.sum()
    pct = counts / total * 100.0
    tbl = pd.DataFrame({"count": counts, "pct": pct})

    # Decide what's kept
    keep_mask = tbl["pct"] >= min_pct
    if include_blank_in_others and "__blank__" in tbl.index:
        keep_mask.loc["__blank__"] = False

    kept = tbl[keep_mask].sort_values("count", ascending=False)
    rest = tbl[~keep_mask]

    # Fold the rest into 'Others'
    others_row = pd.DataFrame(
        {"count": [int(rest["count"].sum())], "pct": [rest["pct"].sum()]},
        index=["Others"],
    )

    final = pd.concat([kept, others_row])
    final["pct"] = final["pct"].round(pct_decimals)

    # Pretty print
    print(final.to_string())

    # Also print a one-line per TLD summary (nice for logs)
    summary_lines = [f"{tld}: {row['count']} ({row['pct']:.{pct_decimals}f}%)"
                     for tld, row in final.iterrows()]
    print("\n" + "\n".join(summary_lines))

    return final


final_tld_table = print_tld_value_counts(
    out,
    url_col="url",
    tld_col="tld",
    min_pct=0.5,                 # fold anything < 0.5% into Others
    include_blank_in_others=True # force '__blank__' into Others
)


In [ ]:
combined = out

In [ ]:
import pandas as pd
import numpy as np

# Define length bins
length_bins = [
    (0, 40),
    (40, 80),
    (80, 100),
    (100, 120),
    (120, 160),
    (160, 200),
    (200, float("inf"))
]

import math

target_total = 1_000_000

# Assign bins
def assign_bin(length):
    for i, (low, high) in enumerate(length_bins):
        if low <= length < high:
            return i
    return None

combined["length_bin"] = combined["url_length"].apply(assign_bin)

# Bin sizes available
bin_groups = combined.groupby("length_bin")
bin_sizes = bin_groups.size().to_dict()
print("Available per bin:", bin_sizes)

# --- Sampling allocation loop ---
remaining_target = target_total
remaining_bins = set(bin_sizes.keys())
allocations = {b: 0 for b in bin_sizes}

while remaining_target > 0 and remaining_bins:
    # Fair share per bin
    share = math.floor(remaining_target / len(remaining_bins))
    filled = set()

    for b in list(remaining_bins):
        capacity = bin_sizes[b] - allocations[b]
        take = min(share, capacity)
        allocations[b] += take
        remaining_target -= take

        if allocations[b] == bin_sizes[b]:  # bin is exhausted
            filled.add(b)

    remaining_bins -= filled

print("Final allocations:", allocations)
print("Total allocated:", sum(allocations.values()))

# --- Draw samples ---
balanced_parts = []
rng = 42
for b, n in allocations.items():
    if n > 0:
        part = bin_groups.get_group(b).sample(n=n, random_state=rng)
        balanced_parts.append(part)

balanced_dataset = pd.concat(balanced_parts, ignore_index=True)

print(f"Balanced dataset size: {len(balanced_dataset)}")
print("Bin counts after balancing:\n", balanced_dataset["length_bin"].value_counts().sort_index())

# print final describe stats
print()

In [ ]:
display(balanced_dataset["url_length"].describe())
# to csv
balanced_dataset = balanced_dataset[["url"]]
balanced_dataset.to_csv("dataset.csv", index=False)